In [ ]:
import scipy
from scipy.spatial.transform import Rotation
import glob
from pathlib import Path
from matplotlib import pyplot as plt
import numpy as np

In [ ]:
# load paths for all EDS sequences
# expects data/processed/<sequence_name>/relative_motions.txt and <sequence_name>.txt
gt_rel_motions_paths = [Path(p) for p in glob.glob('../data/processed/*/relative_motions.txt')]

gt_anchor_paths = [Path(p) for p in glob.glob('../data/processed/*/anchor_poses.txt')]

net_rel_motions_paths = []
for gt_path in gt_rel_motions_paths:
    net_pred = gt_path.parent / (gt_path.parent.name + '.txt')
    net_rel_motions_paths.append(net_pred)

In [ ]:
# dict with sequence name as key and tuple of (gt_rel_motions, net_rel_motions, optimal_scale) as value
seq_data = {}

# calculate scale for minimal RMSE and build pose paths for all sequences
for gt_rel, gt_anchor, net_rel in zip(gt_rel_motions_paths, gt_anchor_paths, net_rel_motions_paths):
    
    # t0_us t1_us t1_us px py pz qx qy qz qw
    gt_rel_motions = np.loadtxt(gt_rel, skiprows=1)[:, 2:5]

    # timestamp_us px py pz qx qy qz qw
    gt_anchors = np.loadtxt(gt_anchor, skiprows=1)
    gt_anchors_pos = gt_anchors[:, 1:4]
    gt_anchors_quat = gt_anchors[:, 4:]

    # t0_us t1_us px py pz sigma_x sigma_y sigma_z
    net_rel_motions = np.loadtxt(net_rel, skiprows=1)[:, 2:5]

    if gt_rel_motions.shape[0] != net_rel_motions.shape[0]:
        raise ValueError('Number of relative motions in gt and net do not match')

    # calculate scale
    mse_scale = np.sum(gt_rel_motions * net_rel_motions) / np.sum(net_rel_motions ** 2)

    # scale minimization using scipy and lambda function based on ATE (absolute trajectory error)
    def ate(gt_anchor_pos, gt_anchor_quat, net_rel_motion, scale):
        net_anochors_pos = [gt_anchor_pos[0, :].astype(np.float64)]
        net_anochors_quat = []
        for i in range(len(gt_anchor_pos)-1):
            # grab current orientation
            net_anochors_quat.append(gt_anchor_quat[i])
            R_curr = Rotation.from_quat(net_anochors_quat[-1])
            
            # take current pose and add on relatie motion in the world frame
            p_next = net_anochors_pos[-1] + R_curr.as_matrix() @ (net_rel_motion[i] * scale)
            net_anochors_pos.append(p_next)
        
        return np.sqrt(np.mean((gt_anchor_pos - np.array(net_anochors_pos)) ** 2))
    
    ate_scale = scipy.optimize.minimize(lambda s: ate(gt_anchors_pos, gt_anchors_quat, net_rel_motions, s), x0=1.0).x[0]

    print(f'Sequence: {gt_rel.parent.name}, RMSE scale: {mse_scale:.4f}, ATE scale: {ate_scale:.4f}')
    
    # build absolute pose trajectories
    net_anochors_pos = [gt_anchors_pos[0].astype(np.float64)]
    net_anochors_quat = [gt_anchors_quat[0].astype(np.float64)]

    # Integrate the poses 
    for i in range(len(gt_rel_motions)-1):
        # grab current orientation
        R_curr = Rotation.from_quat(net_anochors_quat[-1])
        
        # take current pose and add on relatie motion in the world frame
        p_next = net_anochors_pos[-1] + R_curr.as_matrix() @ (net_rel_motions[i])
        net_anochors_pos.append(p_next)
        net_anochors_quat.append(gt_anchors_quat[i+1])
    
    net_anchors = np.concat((gt_anchors[:-1, :1], np.array(net_anochors_pos), np.array(net_anochors_quat)), axis=1)

    seq_data[net_rel.parent.name] = (
        gt_rel_motions,
        gt_anchors,
        net_rel_motions,
        net_anchors,
        mse_scale,
        ate_scale
        )

In [ ]:
# save the data in format usable by scaramuzza's toolbox
for seq in net_rel_motions_paths:
    gt_rel_motions, gt_anchors, net_rel_motions, net_anchors, mse_scale, ate_scale = seq_data[seq.parent.name]

    # t0_us t1_us px py pz sigma_x sigma_y sigma_z
    net_original = np.loadtxt(seq, skiprows=1)

    net_rel_motions_scaled = np.concat((net_original[:, :2],
                                       net_original[:, 2:5] * mse_scale,
                                       net_original[:, 5:]),
                                       axis=1,
                                       )

    np.savetxt(seq.parent / (seq.parent.name + '_scaled_rmse.txt'), net_rel_motions_scaled, delimiter=' ', fmt=f"%d %d {' '.join(['%.10f']*6)}", header='t0_us t1_us px py pz sigma_x sigma_y sigma_z', comments='')
    np.savetxt(seq.parent / ('stamped_traj_estimate.txt'), net_anchors, delimiter=' ', fmt=f"%d {' '.join(['%.10f']*7)}") #, header='timestamp_us px py pz qx qy qz qw', comments='')
    np.savetxt(seq.parent / ('stamped_groundtruth.txt'), gt_anchors, delimiter=' ', fmt=f"%d {' '.join(['%.10f']*7)}") #, header='timestamp_us px py pz qx qy qz qw', comments='')

In [ ]:
for seq_to_inspect in seq_data.keys():
    gt_rel_motions, gt_anchors, net_rel_motions, net_anchors, mse_scale, ate_scale = seq_data[seq_to_inspect]

    old_rmse = np.sqrt(np.mean((gt_rel_motions - net_rel_motions) ** 2))
    new_rmse = np.sqrt(np.mean((gt_rel_motions - ate_scale * net_rel_motions) ** 2))

    print(f'Sequence: {seq_to_inspect}, old RMSE: {old_rmse:.4f}, new RMSE: {new_rmse:.4f}')

    plt.figure(figsize=(10, 8))
    plt.title(f'Sequence: {seq_to_inspect}, old RMSE: {old_rmse:.4f}, new RMSE: {new_rmse:.4f}')
    plt.subplot(3,1,1)
    t = np.arange(gt_rel_motions.shape[0])
    plt.plot(t, gt_rel_motions[:, 0])
    plt.plot(t, net_rel_motions[:, 0])
    plt.plot(t, ate_scale * net_rel_motions[:, 0])
    plt.legend(['gt: ', 'net', f'scaled net {mse_scale:.2f}'])
    plt.xlabel('Time')
    plt.ylabel('rel x')

    plt.subplot(3,1,2)
    plt.plot(t, gt_rel_motions[:, 1])
    plt.plot(t, net_rel_motions[:, 1])
    plt.plot(t, mse_scale * net_rel_motions[:, 1])
    plt.legend(['gt: ', 'net', f'scaled net {mse_scale:.2f}'])
    plt.xlabel('Time')
    plt.ylabel('rel y')

    plt.subplot(3,1,3)
    plt.plot(t, gt_rel_motions[:, 2])
    plt.plot(t, net_rel_motions[:, 2])
    plt.plot(t, mse_scale * net_rel_motions[:, 2])
    plt.legend(['gt: ', 'net', f'scaled net {mse_scale:.2f}'])
    plt.xlabel('Time')
    plt.ylabel('rel z')

    plt.figure()
    plt.plot(gt_anchors[:, 1], gt_anchors[:, 2], label='gt')
    plt.plot(net_anchors[:, 1], net_anchors[:, 2], label='net')
    plt.legend()
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(f'Sequence: {seq_to_inspect}')